<a href="https://colab.research.google.com/github/aspurser84-dot/BIFX546-project/blob/main/notebooks/Supplemental_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Title: Supplemental 2: COVID-19 Variant tracking in California between 2021 through 2023

##Author: Amanda Graham

###Course: BIFX 546 — Machine Learning for Bioinformatics, Spring 2026

###Instructor: Dr. Sarangan (Ravi) Ravichandran

Date: 5/6/26

Disclaimer: This notebook was created as a second supplemental notebook to the Final_project_BIFX546_Graham. This supplemental code uses alternative model building methods to handle imbalanced datasets. These models did not result in the final model but are important to show that they were tested. This notebook uses a publicly available dataset, key concepts from BIFX 546 Machine Learning for Bioinformatics and concepts from the textbook Data Science from Scratch.

#Setting up directory
The code below changes the working directory so the data can be properly saved in Colab.

In [ ]:
import os

# Show current working directory
print("Old directory:", os.getcwd())

# Move up to main directory
os.chdir("/")

print("New directory:", os.getcwd())

Old directory: /content
New directory: /


#Upload the data

Import the dataset from Kagglehub repository. This requires an API key setup in CoLab.

Note: The main notebook shows two methods to upload the dataset. For simplicity, this supplemental notebook uses the direct download method from the Kagglehub repository.

In [ ]:
#Imports the dataset from Kagglehub repository and
#places the dataset in kaggle/input/ folder
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nidzsharma/covid-19-variant-data")

# Verify the actual data file is present, if not force re-download
csv_file = os.path.join(path, "covid19_variant.csv")

if not os.path.exists(csv_file):
  path = kagglehub.dataset_download("nidzsharma/covid-19-variant-data", force_download=True)

print("Path to dataset files:", path)

100%|██████████| 67.0k/67.0k [00:00<00:00, 578kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/nidzsharma/covid-19-variant-data/versions/1


#Data Cleaning

The main notebook provides an overview of exploratory data analysis (EDA) with explanations on why each data cleaning step is performed. See the main notebook for detailed explanations.

The below code converts the downloaded csv file to a pandas dataframe, converts the date format to datetime data type, removes the variables 'area' and 'area_type', removes the rows with the name 'Total' and 'Other' found in the 'variant_name' column and removes rows with zero specimens collected.

In [ ]:
#Converts csv file to pandas dataframe
import pandas as pd

df = pd.read_csv(csv_file)

#converts 'date' variable to datetime data type
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')

#Removes variables 'area' and 'area_type'
df1 = df.copy()
df1 = df1.drop(columns=['area', 'area_type'])

#Remove all rows with "Total" in variant_name
A_value_to_remove = 'Total'
B_value_to_remove = 'Other'

#Remove rows where 'variant_name' equals "Total" and "Other"
df1 = df1[df1['variant_name'] != A_value_to_remove]
df1 = df1[df1['variant_name'] != B_value_to_remove]

# Calculate 7-day rolling average for specimens and percentage
#specimens_7d_avg
data_sorted = df1.sort_values(by= ['variant_name',"date"], ascending=True)
series_specimen = pd.Series(data_sorted['specimens'])

moving_avg_specimens = series_specimen.rolling(window=7).mean()

#percentage_7d_avg
data = df1['percentage'].tolist()
series_percentage = pd.Series(data_sorted['percentage'])

moving_avg_percentage = series_percentage.rolling(window=7).mean()

#Fill in N/A values with 7 day average

df2 = df1.copy()
df2['specimens_7d_avg'] = moving_avg_specimens
df2['percentage_7d_avg'] = moving_avg_percentage

df2['specimens_7d_avg'] = df2['specimens_7d_avg'].fillna(0)
df2['percentage_7d_avg'] = df2['percentage_7d_avg'].fillna(0)

#calculates the rate change for number of specimens per day
#Also fills in the N/A values with zero
rate_change = df2.groupby('variant_name', as_index=False)['specimens'].diff()

df3 = df2.copy()
df3['rate_change'] = rate_change

df3['rate_change'] = df3['rate_change'].fillna(0)

# Calculate 7-day rolling average
#rate change

data_sorted2 = df3.sort_values(by= ['variant_name',"date"], ascending=True)
series_rate = pd.Series(data_sorted2['rate_change'])

moving_avg_rate = series_rate.rolling(window=7).mean()

df3['rate_change_7d_avg'] = moving_avg_rate

df3['rate_change_7d_avg'] = df3['rate_change_7d_avg'].fillna(0)

# Remove rows where there were no specimens collected
df4 = df3.copy()

#Value to drop from a specific column
threshold = 1

#Drop rows where the column matches the value
df5 = df4[df4['specimens'] >= threshold].reset_index(drop=True)
df5 = df5.sort_values(by= ['variant_name',"date"], ascending=True)

print("The original dataset contained", df4.shape, "rows and columns.")
print("The new dataset contains", df5.shape, "rows and columns.")

The original dataset contained (6232, 8) rows and columns.
The new dataset contains (1797, 8) rows and columns.


# Apply threshold on percentage

In [ ]:
# Set a threshold for variables
df6= df5.copy()

# Value to drop from a specific column
threshold = 1

# Drop rows where the column matches the value
df7 = df6[df6['percentage'] >= threshold].reset_index(drop=True)

df7.groupby("variant_name")["specimens"].agg(["min", "max", "median", "mean", "std"])

,min,max,median,mean,std
variant_name,,,,,
Alpha,1,249,64.5,80.745000,66.152405
Beta,1,8,5.0,5.166667,2.037527
Delta,1,3569,731.0,856.056738,847.355285
Epsilon,1,974,112.0,156.427673,168.923526
Gamma,6,58,27.0,25.215385,11.250485
Lambda,2,10,5.0,4.909091,2.427120
Mu,1,41,8.5,14.766667,12.441547
Omicron,1,4961,662.0,870.286364,744.164067


Encode the data

In [ ]:
#encode catagorical values to numeric
#this is needed to preform scalar fit transform

encoder_list = df7['variant_name'].unique()
encoder_values = [i for i in range(len(encoder_list))]
encoder_dict = dict(zip(encoder_list, encoder_values))
df7['variant_name'] = df7['variant_name'].map(encoder_dict)
pd.Series(df7["variant_name"]).value_counts()
print(encoder_list, encoder_values)

['Alpha' 'Beta' 'Delta' 'Epsilon' 'Gamma' 'Lambda' 'Mu' 'Omicron'] [0, 1, 2, 3, 4, 5, 6, 7]


#Random Forest Modeling

#Stratified Split
The code below splits the data stratifying it so there is a balanced representation for each variant.

In [ ]:
from sklearn.model_selection import train_test_split
#Parts of this code were created using Gemini

df_sorted = df7.sort_values(by='date')

#Identify classes with only one member in df_sorted
#credit Gemini
class_counts = df_sorted['variant_name'].value_counts()
single_member_classes = class_counts[class_counts == 1].index

#Filter out rows belonging to single-member classes
#credit Gemini
if not single_member_classes.empty:
    df_sorted = df_sorted[~df_sorted['variant_name'].isin(single_member_classes)].copy()
    print(f"Removed single-member classes from df_sorted: {list(single_member_classes)}")
else:
    print("No single-member classes found to remove.")

y = df_sorted['variant_name'] #Extract the target variable 'variant_name'
X = df_sorted.drop(columns=['date', 'variant_name']) #Extract features (dropping date and variant_name)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

feature_names = X_train.columns.tolist()

print(f'X Training set : {X_train.shape[0]} samples')
print(f'X Test set     : {X_test.shape[0]} samples')

No single-member classes found to remove.
X Training set : 1035 samples
X Test set     : 259 samples


# Random Forest model

In [ ]:
#Create a Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=100, criterion='entropy', oob_score=True,
                             class_weight='balanced',
                             random_state=42)

feature_names = X_train.columns.tolist()

#Fit the model
clf.fit(X_train, y_train)

print(f"Training accuracy : {clf.score(X_train, y_train):.0%}")

#OOB, out of bag score, uses out of bag samples to test the model
#Close to testing accuracy but not identical
print(f"OOB score (≈ test): {clf.oob_score_:.2%}")
print("The Random Forest model was trained on the following features:", feature_names)

from sklearn.metrics import classification_report
#Make predictions
y_pred = clf.predict(X_test)

variant_name = encoder_list

#support number of samples
print(classification_report(y_test, y_pred, labels= encoder_values, target_names= variant_name, zero_division=0 ))

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

#Accuracy, all predictions correct
accuracy = accuracy_score(y_test, y_pred)
print(f"Testing accuracy: {accuracy:.2%}")

#Precision, true positives correctly called
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Precision: {precision:.2%}")

#Recall, true positive rate
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Recall: {recall:.2%}")

#F1 score, model accuracy balanced
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"F1 Score: {f1:.2%}")

Training accuracy : 100%
OOB score (≈ test): 76.62%
The Random Forest model was trained on the following features: ['specimens', 'percentage', 'specimens_7d_avg', 'percentage_7d_avg', 'rate_change', 'rate_change_7d_avg']
              precision    recall  f1-score   support

       Alpha       0.68      0.75      0.71        40
        Beta       0.00      0.00      0.00         2
       Delta       0.70      0.74      0.72        57
     Epsilon       0.81      0.66      0.72        32
       Gamma       0.71      0.77      0.74        26
      Lambda       0.33      0.50      0.40         2
          Mu       0.56      0.75      0.64        12
     Omicron       0.94      0.85      0.89        88

    accuracy                           0.76       259
   macro avg       0.59      0.63      0.60       259
weighted avg       0.78      0.76      0.77       259

Testing accuracy: 76.45%
Precision: 77.80%
Recall: 76.45%
F1 Score: 76.84%


#Balanced Random Forest

In [ ]:
#Balanced random forest model
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from imblearn.ensemble import BalancedRandomForestClassifier

variant_name = encoder_list

# BalancedRandomForest: undersamples the majority class for each tree
brf = BalancedRandomForestClassifier(n_estimators=100, random_state=42)
brf.fit(X_train, y_train)
y_pred_brf = brf.predict(X_test)
y_proba_brf = brf.predict_proba(X_test)

print(f"Training accuracy : {brf.score(X_train, y_train):.0%}")

print("=== BalancedRandomForest ===")
print(classification_report(y_test, y_pred_brf, labels= encoder_values, target_names= variant_name, zero_division=0))

#Accuracy, all predictions correct
accuracy = accuracy_score(y_test, y_pred_brf)
print(f"Testing accuracy: {accuracy:.2%}")

#Precision, true positives correctly called
precision = precision_score(y_test, y_pred_brf, average='weighted', zero_division=0)
print(f"Precision: {precision:.2%}")

#Recall, true positive rate
recall = recall_score(y_test, y_pred_brf, average='weighted', zero_division=0)
print(f"Recall: {recall:.2%}")

#F1 score, model accuracy balanced
f1 = f1_score(y_test, y_pred_brf, average='weighted', zero_division=0)
print(f"F1 Score: {f1:.2%}")

Training accuracy : 72%
=== BalancedRandomForest ===
              precision    recall  f1-score   support

       Alpha       0.58      0.65      0.61        40
        Beta       0.33      0.50      0.40         2
       Delta       0.66      0.44      0.53        57
     Epsilon       0.52      0.41      0.46        32
       Gamma       0.62      0.77      0.69        26
      Lambda       0.25      0.50      0.33         2
          Mu       0.35      0.92      0.51        12
     Omicron       0.89      0.82      0.85        88

    accuracy                           0.65       259
   macro avg       0.53      0.62      0.55       259
weighted avg       0.68      0.65      0.65       259

Testing accuracy: 65.25%
Precision: 68.40%
Recall: 65.25%
F1 Score: 65.48%


# Easy Ensemble

In [ ]:
# EasyEnsemble: trains multiple classifiers on balanced subsets
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
ee = EasyEnsembleClassifier(n_estimators=10, random_state=42)
ee.fit(X_train, y_train)

variant_name = encoder_list

y_pred_ee = ee.predict(X_test)
y_proba_ee = ee.predict_proba(X_test)[:, 1]

print(f"Training accuracy : {ee.score(X_train, y_train):.0%}")

print("=== EasyEnsembleClassifier ===")
print(classification_report(y_test, y_pred_ee, labels= encoder_values, target_names= variant_name, zero_division=0))

#Accuracy, all predictions correct
accuracy = accuracy_score(y_test, y_pred_ee)
print(f"Testing accuracy: {accuracy:.2%}")

#Precision, true positives correctly called
precision = precision_score(y_test, y_pred_ee, average='weighted', zero_division=0)
print(f"Precision: {precision:.2%}")

#Recall, true positive rate
recall = recall_score(y_test, y_pred_ee, average='weighted', zero_division=0)
print(f"Recall: {recall:.2%}")

#F1 score, model accuracy balanced
f1 = f1_score(y_test, y_pred_ee, average='weighted', zero_division=0)
print(f"F1 Score: {f1:.2%}")

Training accuracy : 51%
=== EasyEnsembleClassifier ===
              precision    recall  f1-score   support

       Alpha       0.40      0.55      0.46        40
        Beta       0.33      1.00      0.50         2
       Delta       0.33      0.23      0.27        57
     Epsilon       0.38      0.19      0.25        32
       Gamma       0.53      0.69      0.60        26
      Lambda       0.00      0.00      0.00         2
          Mu       0.41      0.75      0.53        12
     Omicron       0.73      0.70      0.72        88

    accuracy                           0.51       259
   macro avg       0.39      0.51      0.42       259
weighted avg       0.50      0.51      0.49       259

Testing accuracy: 50.97%
Precision: 50.21%
Recall: 50.97%
F1 Score: 49.36%


# SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
print("The original training data contained", X_train.shape, "X training data and", y_train.shape, "y training data")
print("The new training data contains", X_res.shape, "X training data and", y_res.shape, "y training data")

The original training data contained (1035, 6) X training data and (1035,) y training data
The new training data contains (2816, 6) X training data and (2816,) y training data


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# Create a Random Forest Classifier
clf2 = RandomForestClassifier(n_estimators=100, criterion='entropy', oob_score=True,
                             class_weight='balanced',
                             random_state=42)

# Fit the model
clf2.fit(X_res, y_res)

# Make predictions
y_pred2 = clf2.predict(X_test)

variant_name = encoder_list

print(f"Training accuracy : {clf2.score(X_train, y_train):.0%}")

#support number of samples
print(classification_report(y_test, y_pred2, labels= encoder_values, target_names= variant_name, zero_division=0 ))

#Accuracy, all predictions correct
accuracy2 = accuracy_score(y_test, y_pred2)
print(f"Testing accuracy: {accuracy2:.2%}")

#Precision, true positives correctly called
precision2 = precision_score(y_test, y_pred2, average='weighted', zero_division=0)
print(f"Precision: {precision2:.2%}")

#Recall, true positive rate
recall2 = recall_score(y_test, y_pred2, average='weighted', zero_division=0)
print(f"Recall: {recall2:.2%}")

#F1 score, model accuracy balanced
f12 = f1_score(y_test, y_pred2, average='weighted', zero_division=0)
print(f"F1 Score: {f1:.2%}")

Training accuracy : 100%
              precision    recall  f1-score   support

       Alpha       0.68      0.75      0.71        40
        Beta       0.33      0.50      0.40         2
       Delta       0.71      0.79      0.75        57
     Epsilon       0.80      0.62      0.70        32
       Gamma       0.72      0.69      0.71        26
      Lambda       0.20      0.50      0.29         2
          Mu       0.53      0.75      0.62        12
     Omicron       0.96      0.84      0.90        88

    accuracy                           0.76       259
   macro avg       0.62      0.68      0.63       259
weighted avg       0.79      0.76      0.77       259

Testing accuracy: 76.45%
Precision: 78.88%
Recall: 76.45%
F1 Score: 49.36%
